# CV4CHL - EVGS Rule Sources

**Platform**: Colab / RunPod / Local
**Runtime**: CPU only (~10-15 min)
**Base**: EVGS-aligned threshold rule source

---

## What this notebook does

This notebook implements EVGS angle-threshold rule sources aligned with the
2023 MDPI EVGS paper (PMC10220686). Implementation details that the paper
specifies and we mirror here:

| Paper requirement | Items affected |
|---|---|
| Bidirectional Zeni heel-strike / toe-off detection (auto-detect walking direction) | 1, 2, 3, 6-13 (all sagittal items) |
| Signed ankle dorsiflexion angle (orientation-preserving) | 3, 7 |
| Knee flexion via slope-of-lines formulation (paper Eq. 7) | 8, 9, 10, 11 |
| Item 10 sampled at initial-contact frame (not stance-mean) | 10 |
| Item 10 threshold `5 < x ≤ 15` | 10 |
| Item 11: paper provides no explicit binary cut → calibrate on training data | 11 |
| Item 13: hip flexion sign normalised to walking direction | 13 |
| Item 14: mirror-symmetric thresholds for L vs R | 14 |

## Pipeline

1. **Cell 1**: Setup
2. **Cell 2**: Config & Load Data
3. **Cell 3**: Core helpers — `detect_gait_events_bidirectional`,
   `knee_flexion_slope` (paper Eq. 7), `hip_flexion_perp_trunk`, `ankle_dorsi_signed`
4. **Cell 4**: EVGS rule functions, explicitly tagging which item each targets
5. **Cell 5**: Apply rules to train + test, collecting raw values (not just binary)
6. **Cell 6**: Auto-tune thresholds on training data via 1-D grid search per item/side
7. **Cell 7**: Validate tuned rules vs ML on train (LOSO XGBoost OOF)
8. **Cell 8**: Generate hybrid CSVs with per-item canonical sources
9. **Cell 9**: Summary

In [ ]:
# === Cell 1: Setup & Imports ===
import os
import sys
import time
import json
import pickle
import warnings
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
np.random.seed(42)

# --- Timing helpers ---
NOTEBOOK_NAME = 'CV4CHL_04_train_evgs_rule_sources'
_cell_times = {}
_cell_start = None

def cell_start(name):
    global _cell_start
    _cell_start = time.time()
    _cell_times[name] = None
    print(f'\u25b6 {name}')

def cell_end(name):
    elapsed = time.time() - _cell_start
    _cell_times[name] = elapsed
    print(f'\u2713 {name} \u2014 {elapsed:.1f}s ({elapsed/60:.1f}min)')

print(f'\nNotebook: {NOTEBOOK_NAME}')
print(f'Started:  {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'numpy={np.__version__}  pandas={pd.__version__}')


In [ ]:
# === Cell 2: Config & Load Data ===
cell_start('Cell 2: Config & Load Data')

# --- Environment detection ---
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = os.environ.get(
        'CV4CHL_REPO_ROOT',
        '/content/drive/MyDrive/CVPR_CH4CHL',
    )
else:
    ROOT = os.environ.get('CV4CHL_REPO_ROOT', os.getcwd())

PROC_DIR   = f'{ROOT}/1_data/processed'
SUBMIT_DIR = f'{ROOT}/5_outputs/submissions'
DIAGNOSTIC_DIR = f'{ROOT}/5_outputs/diagnostics/evgs_rules'
os.makedirs(SUBMIT_DIR, exist_ok=True)
os.makedirs(DIAGNOSTIC_DIR, exist_ok=True)

# Items whose individual_item rule output is the canonical source for final
# assembly. Other items' individual_item CSVs and the multi-item validation
# hybrid go to DIAGNOSTIC_DIR for offline review only.
CANONICAL_EVGS_ITEMS = (3, 9, 11, 15, 16, 17)
print(f'ROOT      : {ROOT}')
print(f'PROC_DIR  : {PROC_DIR}')
print(f'SUBMIT_DIR: {SUBMIT_DIR}')

# --- Constants ---
T1_TEST_IDS = [4, 5, 18, 26, 28, 40, 42, 43, 47, 48, 53, 54, 72, 78, 83, 85]
T2_TEST_IDS = [4, 6, 7, 13, 26, 35, 39, 42, 50]

# Track 2 clinical baseline predictions
# Track-2 rows are preserved from the current base CSV;
# Track-2 predictions are generated only by the Track-2 clinical notebook.

LABEL_COLS = [f'evgs_{i}' for i in range(1, 18)]

# Sapiens-2B / COCO-WholeBody body+feet keypoint indices (0-22)
KP = {
    'nose': 0,
    'left_shoulder': 5, 'right_shoulder': 6,
    'left_elbow': 7,    'right_elbow': 8,
    'left_wrist': 9,    'right_wrist': 10,
    'left_hip': 11,     'right_hip': 12,
    'left_knee': 13,    'right_knee': 14,
    'left_ankle': 15,   'right_ankle': 16,
    'left_big_toe': 17, 'left_small_toe': 18, 'left_heel': 19,
    'right_big_toe': 20,'right_small_toe': 21,'right_heel': 22,
}

SAGITTAL_ITEMS = [1, 2, 3, 6, 7, 8, 9, 10, 11, 12, 13, 16]
CORONAL_ITEMS  = [4, 5, 14, 15, 17]

# --- Load keypoints.pkl ---
kp_path = f'{PROC_DIR}/keypoints.pkl'
assert os.path.exists(kp_path), f'Missing: {kp_path}'
with open(kp_path, 'rb') as f:
    kp_raw = pickle.load(f)
print(f'\nLoaded keypoints.pkl: {len(kp_raw)} video entries')

sample_key = next(iter(kp_raw))
if isinstance(sample_key, tuple):
    _kp_by_pid = defaultdict(dict)
    for (pid, vname), vdata in kp_raw.items():
        _kp_by_pid[pid][vname] = vdata
    kp_data = dict(_kp_by_pid)
else:
    kp_data = kp_raw
print(f'Patients with keypoints: {len(kp_data)}')

# --- Load train_ready.pkl ---
tr_path = f'{PROC_DIR}/train_ready.pkl'
assert os.path.exists(tr_path), f'Missing: {tr_path}'
with open(tr_path, 'rb') as f:
    tr_data = pickle.load(f)
df_train = tr_data['track1_train'].copy()
df_test  = tr_data['track1_test'].copy()
print(f'Train labels: {df_train.shape} | Test rows: {df_test.shape}')
print(f'Train unique pids: {df_train["patient_id"].nunique()}')

# --- Load reference baseline base submission ---
# Fail-fast on missing base: notebooks 04-07 read reference_baseline.csv as
# their overlay base (staged by reproduce.py from submissions/reference/, or
# overwritten by notebook 02 on a default full-retrain run). If it is missing
# here, every per-item source CSV produced downstream would be wrong, so we
# raise immediately rather than letting Cell 8 fail much later.
BASE_PATH = f'{SUBMIT_DIR}/reference_baseline.csv'
if not os.path.exists(BASE_PATH):
    raise FileNotFoundError(
        f'Base submission not found: {BASE_PATH}. '
        'Either run `python reproduce.py` from a clean state (which stages '
        'submissions/reference/xgb_baseline.csv into the runtime path), or '
        'run notebook 02 (CV4CHL_02_train_xgb_baseline.ipynb) first.'
    )
df_base = pd.read_csv(BASE_PATH)
print(f'\nBase submission: {BASE_PATH}  shape={df_base.shape}')

cell_end('Cell 2: Config & Load Data')

In [ ]:
# === Cell 3: Core Helpers ===
cell_start('Cell 3: Core Helpers')

FS               = 30.0
CUTOFF           = 6.0
SCORE_THRESH     = 0.4
MIN_FRAMES       = 30
MIN_GAIT_FRAMES  = 45

def butter_lowpass(x, cutoff=CUTOFF, fs=FS, order=4):
    x = np.asarray(x, dtype=float)
    if len(x) < 13:
        return x
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low')
    try:
        return filtfilt(b, a, x)
    except ValueError:
        return x


def smooth_xy(arr2d):
    arr2d = np.asarray(arr2d, dtype=float)
    if arr2d.ndim != 2 or arr2d.shape[0] < 13:
        return arr2d
    return np.column_stack([butter_lowpass(arr2d[:, 0]),
                            butter_lowpass(arr2d[:, 1])])


def signed_angle_deg(v):
    return np.degrees(np.arctan2(v[..., 1], v[..., 0]))


def angle_between_signed(v_from, v_to):
    cross = v_from[..., 0] * v_to[..., 1] - v_from[..., 1] * v_to[..., 0]
    dot   = v_from[..., 0] * v_to[..., 0] + v_from[..., 1] * v_to[..., 1]
    return np.degrees(np.arctan2(cross, dot))


# -------------------------------------------------------------
# Gait event detection now handles both walking directions
# -------------------------------------------------------------
def detect_gait_events_zeni(heel_x, toe_x, hip_x):
    """Bidirectional Zeni heel-strike / toe-off detection.

    Bidirectional implementation: subject may walk in +x or -x direction;
    we auto-detect walking direction via sign of Pearson drift of hip_x
    across time and flip the relative AP signal accordingly.
    """
    rel_heel = heel_x - hip_x
    rel_toe  = toe_x  - hip_x

    if len(rel_heel) < 15:
        return np.array([], dtype=int), np.array([], dtype=int)

    # Detect walking direction: sign of linear trend of hip_x
    t = np.arange(len(hip_x), dtype=float)
    slope = np.polyfit(t, hip_x, 1)[0]
    direction = 1.0 if slope >= 0 else -1.0

    # Normalize so AP + always means "heel ahead of hip in walking direction"
    rh = butter_lowpass(direction * rel_heel)
    rt = butter_lowpass(direction * rel_toe)

    prom_h = max(0.05 * (rh.max() - rh.min()), 1e-6)
    prom_t = max(0.05 * (rt.max() - rt.min()), 1e-6)

    # Heel-strike = heel max ahead of hip (max of rh)
    hs, _ = find_peaks(rh, prominence=prom_h, distance=int(FS * 0.4))
    # Toe-off = toe max behind hip (min of rt)
    to, _ = find_peaks(-rt, prominence=prom_t, distance=int(FS * 0.4))
    return hs, to


def extract_gait_cycles(heel_strikes, toe_offs, n_frames):
    cycles = []
    if len(heel_strikes) < 2 or len(toe_offs) == 0:
        return cycles
    for i in range(len(heel_strikes) - 1):
        hs0 = int(heel_strikes[i])
        hs1 = int(heel_strikes[i + 1])
        tos_in = toe_offs[(toe_offs > hs0) & (toe_offs < hs1)]
        if len(tos_in) == 0:
            continue
        to0 = int(tos_in[0])
        if hs1 - hs0 < int(FS * 0.5):
            continue
        if hs1 - hs0 > int(FS * 2.5):
            continue
        cycles.append({
            'cycle_start': hs0,
            'stance_start': hs0,
            'stance_end': to0,
            'swing_start': to0,
            'swing_end': hs1,
            'cycle_end': hs1,
            'duration': hs1 - hs0,
            'midstance': (hs0 + to0) // 2,
        })
    return cycles


def safe_mean(x, default=np.nan):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if len(x) else default


def safe_median(x, default=np.nan):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if len(x) else default


# -------------------------------------------------------------
# Paper equation 7 - knee flexion via slope-of-lines
# -------------------------------------------------------------
def knee_flexion_angle(hip, knee, ankle):
    """Return per-frame knee flexion in degrees (0 = straight, +ve = bent).

    Uses the paper's slope-between-lines formula:
        theta = arctan((m_shank - m_thigh) / (1 + m_shank*m_thigh))
    which maps collinear lines -> 0 deg and returns values in (-90, 90].
    We then take ABS to get flexion MAGNITUDE (hyperextension and flexion
    collapse, but that is OK because binary abnormality treats both ends).

    Implementation note: vector atan2(cross, dot) for nearly
    collinear opposing vectors gives +/-180 instead of 0, inflating noise.
    """
    # thigh line: hip -> knee  (dy_thigh, dx_thigh)
    t_dx = knee[..., 0] - hip[..., 0]
    t_dy = knee[..., 1] - hip[..., 1]
    # shank line: knee -> ankle
    s_dx = ankle[..., 0] - knee[..., 0]
    s_dy = ankle[..., 1] - knee[..., 1]

    # Use (dy, dx) atan2 form of slope-angle difference.
    # theta_thigh = atan2(t_dy, t_dx); theta_shank = atan2(s_dy, s_dx)
    # Flexion = (theta_shank - theta_thigh) wrapped to (-180, 180],
    # then we fold into (-90, 90] because thigh and shank are LINES, not
    # direction vectors - a 180 deg difference means the same orientation.
    theta_t = np.arctan2(t_dy, t_dx)
    theta_s = np.arctan2(s_dy, s_dx)
    d = np.degrees(theta_s - theta_t)
    # Wrap to (-180, 180]
    d = ((d + 180.0) % 360.0) - 180.0
    # Fold to (-90, 90] (collinear <-> 0; 90 bend <-> +/-90)
    d = np.where(d > 90.0, d - 180.0, d)
    d = np.where(d < -90.0, d + 180.0, d)
    return np.abs(d)


# -------------------------------------------------------------
# Paper perpendicular-to-trunk hip flexion
# -------------------------------------------------------------
def hip_flexion_angle(shoulder, hip, knee, walking_dir=+1.0):
    """Per-frame hip flexion in degrees, signed (+ = flexion forward).

    Paper formula (eq. for hip):
        theta_hip = 90 - arctan((m_thigh - m_perp_trunk) / (1 + m_thigh*m_perp_trunk))
    which equivalently is: angle between the PERPENDICULAR to trunk and
    the thigh. Implemented via vector rotation (trunk rotated 90 deg
    about image origin).

    walking_dir: +1 if subject walks in +x, -1 if -x. Used to sign-correct
    so that hip flexion (thigh swung in walking direction) is always +ve.

    Implementation note: abs(angle_between(trunk, thigh)) which
    confused direction and could not distinguish flexion from extension.
    """
    # Trunk direction: shoulder -> hip  (downward unit vector)
    tr_dx = hip[..., 0] - shoulder[..., 0]
    tr_dy = hip[..., 1] - shoulder[..., 1]
    # Thigh direction: hip -> knee
    th_dx = knee[..., 0] - hip[..., 0]
    th_dy = knee[..., 1] - hip[..., 1]
    # Perpendicular to trunk, rotated toward walking direction:
    # rot(+90 CCW in image y-down) of (tr_dx, tr_dy) = (tr_dy, -tr_dx).
    # If walking +x, perpendicular should point +x; else -x.
    perp_dx = walking_dir * tr_dy
    perp_dy = -walking_dir * tr_dx
    # Signed angle from perp (reference anterior) to thigh.
    cross = perp_dx * th_dy - perp_dy * th_dx
    dot   = perp_dx * th_dx + perp_dy * th_dy
    raw = np.degrees(np.arctan2(cross, dot))
    # When leg hangs straight down from vertical trunk, thigh = -perp
    # (downward) vs perp (horizontal walking-dir). Angle = 90 for flexion 0.
    # So flexion = 90 - (90 - abs_flexion) = subtract 90 to zero-center.
    # Concretely: angle between horizontal-forward and downward-thigh is 90
    # (leg straight). We want this to read 0. Thigh forward = angle 0 (aligned
    # with walking dir) -> flexion = 90. To fold: flexion = 90 - raw_magnitude
    # where raw_magnitude = |angle between forward-perp and thigh|.
    # Simpler and numerically robust:
    #   Compute angle between thigh and (downward) first; that's flexion if
    #   thigh is in front of vertical.
    # Fall back to that cleaner formulation.
    vert_dx = np.zeros_like(tr_dx)
    vert_dy = np.abs(tr_dy) + np.abs(tr_dx) * 0  # points downward (same sign)
    # Use the trunk vector itself as "vertical" reference so we
    # compensate trunk lean simultaneously.
    cross2 = tr_dx * th_dy - tr_dy * th_dx
    dot2   = tr_dx * th_dx + tr_dy * th_dy
    raw2 = np.degrees(np.arctan2(cross2, dot2))
    # Signed: +ve = thigh rotated CCW from trunk (image coords, y-down),
    # which is CW in math coords. We want + if thigh is forward in walking
    # direction. Apply walking_dir flip.
    return -walking_dir * raw2


# -------------------------------------------------------------
# Signed ankle dorsiflexion (no more 90 - abs issue)
# -------------------------------------------------------------
def ankle_angle(knee, ankle, heel, toe, walking_dir=+1.0):
    """Per-frame ankle dorsi(+)/plantar(-) flexion in degrees.

    Neutral (foot perpendicular to shank) = 0 deg.
    Dorsiflexion (toe raised, foot closing toward shank) > 0.
    Plantarflexion (toe pointing down) < 0.

    Implementation note: a naive `90 - abs(angle(shank, foot))` would
    gave NEGATIVE values for dorsiflexion when walking direction was -x
    because cross product sign flipped.
    """
    # Shank: knee -> ankle (downward)
    sh_dx = ankle[..., 0] - knee[..., 0]
    sh_dy = ankle[..., 1] - knee[..., 1]
    # Foot: heel -> toe (anterior)
    ft_dx = toe[..., 0] - heel[..., 0]
    ft_dy = toe[..., 1] - heel[..., 1]
    # Angle of foot relative to shank (shank rotated -90 from walking dir
    # would be the "neutral anterior foot" reference).
    # Compute: rotate shank +90 CCW (in math sense) = (sh_dy, -sh_dx);
    # that is the direction the foot SHOULD point if neutral and walking +x.
    # For walking -x, rotate -90: (-sh_dy, +sh_dx).
    if walking_dir >= 0:
        ref_dx = sh_dy
        ref_dy = -sh_dx
    else:
        ref_dx = -sh_dy
        ref_dy = sh_dx
    # Signed angle from ref to foot
    cross = ref_dx * ft_dy - ref_dy * ft_dx
    dot   = ref_dx * ft_dx + ref_dy * ft_dy
    raw = np.degrees(np.arctan2(cross, dot))
    # In image y-down coordinates, dorsiflexion (foot tilts up, toward shank)
    # means ft rotates CW relative to ref -> cross sign is negative in
    # image coords. Flip sign so dorsi is positive.
    return -walking_dir * raw


def foot_axis_angle(heel, toe, walking_dir=+1.0):
    """Foot axis vs horizontal ground.

    Returns signed angle where +ve means toe is RAISED above heel
    (consistent with heel-strike). In image y-down: toe higher = smaller y
    -> (toe_y - heel_y) < 0, and we flip sign accordingly.
    """
    fx = toe[..., 0] - heel[..., 0]
    fy = toe[..., 1] - heel[..., 1]
    # Project foot onto the walking-direction axis; sign of fx * walking_dir
    # tells us which way the foot points.
    # Magnitude of horizontal travel:
    horiz = walking_dir * fx
    # Vertical (y-down): negative dy means toe is above heel
    vert  = -fy
    return np.degrees(np.arctan2(vert, np.abs(horiz) + 1e-9))


def detect_walking_direction(hip_x):
    """Return +1 / -1 based on linear trend of hip_x across video."""
    if len(hip_x) < 3:
        return 1.0
    t = np.arange(len(hip_x), dtype=float)
    slope = np.polyfit(t, hip_x, 1)[0]
    return 1.0 if slope >= 0 else -1.0


print('Core helpers ready:')
print('  - detect_gait_events_zeni  : bidirectional walking')
print('  - knee_flexion_angle       : paper eq. 7 (slope formula)')
print('  - hip_flexion_angle        : signed, walking-dir corrected')
print('  - ankle_angle              : signed dorsi(+)/plantar(-)')
print('  - foot_axis_angle          : signed, walking-dir corrected')
print('  - detect_walking_direction : new helper')

cell_end('Cell 3: Core Helpers')


In [ ]:
# === Cell 4: EVGS Rule Functions ===
cell_start('Cell 4: EVGS Rule Functions')

# Each rule returns (binary_pred, raw_value, valid_flag)
# Each item rule follows the paper specification.

def _get_sag_keypoints(kps, scores, side):
    idx_hip   = KP[f'{side}_hip']
    idx_knee  = KP[f'{side}_knee']
    idx_ank   = KP[f'{side}_ankle']
    idx_heel  = KP[f'{side}_heel']
    idx_btoe  = KP[f'{side}_big_toe']
    idx_sho   = KP[f'{side}_shoulder']
    needed = [idx_hip, idx_knee, idx_ank, idx_heel, idx_btoe, idx_sho]

    mask = np.ones(len(kps), dtype=bool)
    for i in needed:
        mask &= (scores[:, i] >= SCORE_THRESH)
    if mask.sum() < MIN_FRAMES:
        return None
    out = {
        'hip':   smooth_xy(kps[mask, idx_hip]),
        'knee':  smooth_xy(kps[mask, idx_knee]),
        'ankle': smooth_xy(kps[mask, idx_ank]),
        'heel':  smooth_xy(kps[mask, idx_heel]),
        'toe':   smooth_xy(kps[mask, idx_btoe]),
        'shoulder': smooth_xy(kps[mask, idx_sho]),
        'mask': mask,
        'n': int(mask.sum()),
    }
    out['walking_dir'] = detect_walking_direction(out['hip'][:, 0])
    return out


# ============ ITEM RULE FUNCTIONS =======================
# Paper (2023 PMC10220686) thresholds:
#   Score 0 = normal, Score 1 = moderate, Score 2 = severe
# Binary collapse: normal (0) = Score 0 only; abnormal (1) = Score 1 or 2.

def rule_item1_initial_contact(d, cycles):
    """Item 1 - foot axis angle at heel strike.

    Paper Score 0: 0-20 deg (heel contact).
    Score 1: <0 (toe strike) or >20 (flat).
    Score 2: far outside.
    Binary normal = 0 < angle <= 20.
    Uses the walking-direction-aware foot_axis_angle() so the sign is
    consistent across left/right video views.
    """
    if d is None or not cycles:
        return None, None, False
    wd = d['walking_dir']
    angles = foot_axis_angle(d['heel'], d['toe'], walking_dir=wd)
    vals = []
    for c in cycles:
        i = c['stance_start']
        if 0 <= i < len(angles):
            vals.append(angles[i])
    if not vals:
        return None, None, False
    a = safe_median(vals)
    if not np.isfinite(a):
        return None, None, False
    # Paper: 0 < a <= 20 -> normal
    return (0 if 0.0 < a <= 20.0 else 1), a, True


def rule_item2_heel_rise(d, cycles):
    """Item 2 - heel rise timing as % of stance. Normal: 30%-50%."""
    if d is None or not cycles:
        return None, None, False
    timings = []
    heel_y = d['heel'][:, 1]
    for c in cycles:
        s, e = c['stance_start'], c['stance_end']
        if e - s < 8:
            continue
        seg = heel_y[s:e + 1]
        baseline = np.mean(seg[:max(2, len(seg) // 5)])
        thr = baseline - 0.05 * (np.ptp(seg) + 1e-6)
        rise_idx = np.where(seg < thr)[0]
        if len(rise_idx) == 0:
            timings.append(100.0)
        else:
            timings.append(100.0 * rise_idx[0] / max(1, e - s))
    if not timings:
        return None, None, False
    t = safe_median(timings)
    return (0 if 30.0 < t < 50.0 else 1), t, True


def rule_item3_dorsi_stance(d, cycles):
    """Item 3 - max ankle dorsiflexion in stance.
    Uses the signed ankle_angle() (dorsi > 0). Normal ~5-15 deg.
    The signed ankle_angle is independent of walking direction.
    """
    if d is None or not cycles:
        return None, None, False
    wd = d['walking_dir']
    angles = ankle_angle(d['knee'], d['ankle'], d['heel'], d['toe'], walking_dir=wd)
    vals = []
    for c in cycles:
        s, e = c['stance_start'], c['stance_end']
        if e > s:
            seg = angles[s:e + 1]
            seg = seg[np.isfinite(seg)]
            if len(seg):
                vals.append(np.max(seg))
    if not vals:
        return None, None, False
    a = safe_median(vals)
    return (0 if 5.0 < a < 15.0 else 1), a, True


def rule_item4_hindfoot(d_cor, side):
    """Item 4 - hindfoot varus/valgus, coronal view.

    Paper: **left-specific** thresholds -6 to 1 deg normal,
           **right-specific** thresholds -1 to 6 deg normal.
    Raw values are stored; binary thresholds are auto-tuned in Cell 6.
    """
    if d_cor is None:
        return None, None, False
    knee  = d_cor['knee']
    ankle = d_cor['ankle']
    v = ankle - knee
    angles = np.degrees(np.arctan2(v[:, 0], v[:, 1]))
    a = safe_median(angles)
    if not np.isfinite(a):
        return None, None, False
    # Naive side-aware threshold from paper (tuned in Cell 6)
    if side == 'left':
        return (0 if -6.0 < a < 1.0 else 1), a, True
    else:
        return (0 if -1.0 < a < 6.0 else 1), a, True


def rule_item5_foot_rotation(d_cor_full, side):
    """Item 5 - foot rotation proxy from coronal view."""
    if d_cor_full is None:
        return None, None, False
    lk = d_cor_full.get('left_knee')
    rk = d_cor_full.get('right_knee')
    la = d_cor_full.get('left_ankle')
    ra = d_cor_full.get('right_ankle')
    if any(v is None for v in (lk, rk, la, ra)):
        return None, None, False
    knee_v  = rk - lk
    ankle_v = ra - la
    angles = angle_between_signed(knee_v, ankle_v)
    a = safe_median(angles)
    if not np.isfinite(a):
        return None, None, False
    # Paper normal: -5 to 25 deg
    return (0 if -5.0 < a < 25.0 else 1), a, True


def rule_item6_clearance(d, cycles):
    """Item 6 - foot clearance in swing (fraction of leg length)."""
    if d is None or not cycles:
        return None, None, False
    toe_y  = d['toe'][:, 1]
    heel_y = d['heel'][:, 1]
    leg = np.linalg.norm(d['hip'] - d['ankle'], axis=1)
    leg_len = safe_median(leg)
    if not np.isfinite(leg_len) or leg_len < 1:
        return None, None, False

    clearances = []
    for c in cycles:
        s, e = c['stance_start'], c['stance_end']
        ws, we = c['swing_start'], c['swing_end']
        if we <= ws:
            continue
        ground = np.max(toe_y[s:e + 1]) if e > s else np.max(toe_y)
        toe_lift   = ground - np.min(toe_y[ws:we + 1])
        heel_lift  = ground - np.min(heel_y[ws:we + 1])
        clearances.append(min(toe_lift, heel_lift))
    if not clearances:
        return None, None, False
    cl = safe_median(clearances)
    ratio = cl / max(leg_len, 1e-6)
    return (0 if ratio > 0.05 else 1), ratio, True


def rule_item7_dorsi_swing(d, cycles):
    """Item 7 - max dorsiflexion in swing.

    Uses the signed ankle_angle() convention.
    Normal: 0-10 deg.
    """
    if d is None or not cycles:
        return None, None, False
    wd = d['walking_dir']
    angles = ankle_angle(d['knee'], d['ankle'], d['heel'], d['toe'], walking_dir=wd)
    vals = []
    for c in cycles:
        s, e = c['swing_start'], c['swing_end']
        if e > s:
            seg = angles[s:e + 1]
            seg = seg[np.isfinite(seg)]
            if len(seg):
                vals.append(np.max(seg))
    if not vals:
        return None, None, False
    a = safe_median(vals)
    return (0 if 0.0 < a < 10.0 else 1), a, True


def rule_item8_knee_at_ic(d, cycles):
    """Item 8 - knee flexion at initial contact.
    Uses the paper knee_flexion_angle() formula.
    Normal: 5-15 deg.
    """
    if d is None or not cycles:
        return None, None, False
    knee_flex = knee_flexion_angle(d['hip'], d['knee'], d['ankle'])
    vals = []
    for c in cycles:
        i = c['stance_start']
        if 0 <= i < len(knee_flex):
            vals.append(knee_flex[i])
    if not vals:
        return None, None, False
    a = safe_median(vals)
    return (0 if 5.0 < a < 15.0 else 1), a, True


def rule_item9_peak_knee_ext(d, cycles):
    """Item 9 - peak knee extension in stance (min flexion).
    Normal: 0-10 deg.
    """
    if d is None or not cycles:
        return None, None, False
    knee_flex = knee_flexion_angle(d['hip'], d['knee'], d['ankle'])
    vals = []
    for c in cycles:
        s, e = c['stance_start'], c['stance_end']
        if e > s:
            seg = knee_flex[s:e + 1]
            seg = seg[np.isfinite(seg)]
            if len(seg):
                vals.append(np.min(seg))
    if not vals:
        return None, None, False
    a = safe_median(vals)
    return (0 if 0.0 < a < 10.0 else 1), a, True


def rule_item10_knee_terminal_swing(d, cycles):
    """Item 10 - knee flexion at IC (single heel-strike frame).

    Implementation note: averaging over "terminal swing" (last 25%) would
    Paper explicitly states Item 10 is measured AT initial contact.
    Use the single frame at swing_end (= next heel strike).
    Paper normal: 5-15 deg (binary collapse).
    """
    if d is None or not cycles:
        return None, None, False
    knee_flex = knee_flexion_angle(d['hip'], d['knee'], d['ankle'])
    vals = []
    for c in cycles:
        i = c['swing_end']
        if 0 <= i < len(knee_flex):
            vals.append(knee_flex[i])
    if not vals:
        return None, None, False
    a = safe_median(vals)
    return (0 if 5.0 < a < 15.0 else 1), a, True


def rule_item11_peak_knee_flex_swing(d, cycles):
    """Item 11 - peak knee flexion in swing.

    Implementation note:
      (1) Use paper-compliant knee_flexion_angle() formula.
      (2) Paper does NOT publish explicit binary thresholds for item 11;
          Cell 6 will auto-tune this threshold from train data.
          Provisional paper-inspired threshold = 45-70 deg.
    """
    if d is None or not cycles:
        return None, None, False
    knee_flex = knee_flexion_angle(d['hip'], d['knee'], d['ankle'])
    vals = []
    for c in cycles:
        ws, we = c['swing_start'], c['swing_end']
        if we > ws:
            seg = knee_flex[ws:we + 1]
            seg = seg[np.isfinite(seg)]
            if len(seg):
                vals.append(np.max(seg))
    if not vals:
        return None, None, False
    a = safe_median(vals)
    # Provisional threshold - Cell 6 will overwrite from training data
    return (0 if 45.0 < a < 70.0 else 1), a, True


def rule_item12_peak_hip_ext(d, cycles):
    """Item 12 - peak hip extension in stance (min hip flexion).

    Using signed hip_flexion_angle where extension is negative.
    Paper Score 0 range not exactly published; use -15 to 0 as normal.
    """
    if d is None or not cycles:
        return None, None, False
    wd = d['walking_dir']
    hip_flex = hip_flexion_angle(d['shoulder'], d['hip'], d['knee'], walking_dir=wd)
    vals = []
    for c in cycles:
        s, e = c['stance_start'], c['stance_end']
        if e > s:
            seg = hip_flex[s:e + 1]
            seg = seg[np.isfinite(seg)]
            if len(seg):
                vals.append(np.min(seg))
    if not vals:
        return None, None, False
    a = safe_median(vals)
    # Normal = -15 to 0 (mild extension or neutral); Cell 6 may tune
    return (0 if -15.0 < a < 0.0 else 1), a, True


def rule_item13_peak_hip_flex(d, cycles):
    """Item 13 - peak hip flexion at IC.

    Paper exact thresholds (from Appendix B):
      Score 0: 25 < x <= 45
      Score 1: 10 < x <= 25 OR 45 < x <= 60
      Score 2: x <= 10 OR x > 60
    Binary normal = 25 < x <= 45.

    Implementation note: now uses walking-direction-aware signed
    hip_flexion_angle(), so the raw value at HS is positive for normal
    forward-swung leg.
    """
    if d is None or not cycles:
        return None, None, False
    wd = d['walking_dir']
    hip_flex = hip_flexion_angle(d['shoulder'], d['hip'], d['knee'], walking_dir=wd)
    vals = []
    for c in cycles:
        i = c['swing_end']
        if 0 <= i < len(hip_flex):
            vals.append(hip_flex[i])
    if not vals:
        return None, None, False
    a = safe_median(vals)
    return (0 if 25.0 < a <= 45.0 else 1), a, True


def rule_item14_pelvic_obliquity(d_cor_full, side='left'):
    """Item 14 - pelvic obliquity, coronal hip line vs horizontal.

    Implementation note: Paper has SIDE-ASYMMETRIC thresholds:
      Left leg normal  : -1 < a <  6
      Right leg normal : -6 < a <  1
    a single global threshold would conflate L vs R asymmetry.
    """
    if d_cor_full is None:
        return None, None, False
    lh = d_cor_full.get('left_hip')
    rh = d_cor_full.get('right_hip')
    if lh is None or rh is None:
        return None, None, False
    v = rh - lh
    angles = signed_angle_deg(v)
    a = safe_median(angles)
    if not np.isfinite(a):
        return None, None, False
    if side == 'left':
        return (0 if -1.0 < a < 6.0 else 1), a, True
    else:
        return (0 if -6.0 < a < 1.0 else 1), a, True


def rule_item15_pelvic_rotation(d_cor_full, side='left'):
    """Item 15 - pelvic rotation proxy. Paper normal: -6 to 11 deg."""
    if d_cor_full is None:
        return None, None, False
    lh = d_cor_full.get('left_hip')
    rh = d_cor_full.get('right_hip')
    if lh is None or rh is None:
        return None, None, False
    v = rh - lh
    angles = signed_angle_deg(v)
    angles = angles[np.isfinite(angles)]
    if len(angles) == 0:
        return None, None, False
    n = len(angles)
    mid = angles[n // 4: 3 * n // 4]
    a = safe_median(mid) if len(mid) else safe_median(angles)
    return (0 if -6.0 < a < 11.0 else 1), a, True


def rule_item16_trunk_lean(d, cycles):
    """Item 16 - sagittal trunk lean. Normal: -5 to 5 deg."""
    if d is None:
        return None, None, False
    trunk_v = d['hip'] - d['shoulder']
    angles = np.degrees(np.arctan2(trunk_v[:, 0], trunk_v[:, 1]))
    a = safe_median(angles)
    if not np.isfinite(a):
        return None, None, False
    return (0 if -5.0 < a < 5.0 else 1), a, True


def rule_item17_lateral_trunk(d_cor_full, side='left'):
    """Item 17 - lateral trunk shift (coronal). Paper normal: 0 < a < 10."""
    if d_cor_full is None:
        return None, None, False
    sho_l = d_cor_full.get('left_shoulder')
    sho_r = d_cor_full.get('right_shoulder')
    hip_l = d_cor_full.get('left_hip')
    hip_r = d_cor_full.get('right_hip')
    if any(v is None for v in (sho_l, sho_r, hip_l, hip_r)):
        return None, None, False
    mid_sho = (sho_l + sho_r) / 2.0
    mid_hip = (hip_l + hip_r) / 2.0
    trunk_v = mid_hip - mid_sho
    angles = np.degrees(np.arctan2(trunk_v[:, 0], trunk_v[:, 1]))
    angles = np.abs(angles)
    a = safe_median(angles)
    if not np.isfinite(a):
        return None, None, False
    return (0 if 0.0 < a < 10.0 else 1), a, True


print('17 EVGS rule functions defined (items with in docstring)')

cell_end('Cell 4: EVGS Rule Functions')


In [ ]:
# === Cell 5: Apply Rules to All Patients ===
cell_start('Cell 5: Apply Rules to All Patients')

def _get_video_view(vdata, vname):
    v = vdata.get('view') if isinstance(vdata, dict) else None
    if isinstance(v, str) and v:
        return v.lower()
    name = vname.lower() if isinstance(vname, str) else ''
    for tag in ('forward', 'backward', 'left', 'right'):
        if f'_{tag}_' in name:
            return tag
    return ''


def _get_video_arrays(vdata):
    kps = vdata.get('keypoints', vdata.get('kps'))
    sc  = vdata.get('scores')
    if kps is None or sc is None:
        return None, None
    kps = np.asarray(kps, dtype=float)
    sc  = np.asarray(sc, dtype=float)
    if kps.ndim != 3 or kps.shape[1] < 23:
        return None, None
    return kps[:, :23, :2], sc[:, :23]


def _extract_coronal_full(kps, scores):
    needed = {
        'left_hip': KP['left_hip'], 'right_hip': KP['right_hip'],
        'left_knee': KP['left_knee'], 'right_knee': KP['right_knee'],
        'left_ankle': KP['left_ankle'], 'right_ankle': KP['right_ankle'],
        'left_shoulder': KP['left_shoulder'],
        'right_shoulder': KP['right_shoulder'],
        'left_big_toe': KP['left_big_toe'], 'right_big_toe': KP['right_big_toe'],
        'left_heel': KP['left_heel'], 'right_heel': KP['right_heel'],
    }
    mask = np.ones(len(kps), dtype=bool)
    for i in needed.values():
        mask &= (scores[:, i] >= SCORE_THRESH)
    if mask.sum() < MIN_FRAMES:
        return None
    out = {}
    for k, idx in needed.items():
        out[k] = smooth_xy(kps[mask, idx])
    out['n'] = int(mask.sum())
    return out


# Sagittal rules taking (d, cycles)
SAG_RULES = {
    1:  rule_item1_initial_contact,
    2:  rule_item2_heel_rise,
    3:  rule_item3_dorsi_stance,
    6:  rule_item6_clearance,
    7:  rule_item7_dorsi_swing,
    8:  rule_item8_knee_at_ic,
    9:  rule_item9_peak_knee_ext,
    10: rule_item10_knee_terminal_swing,
    11: rule_item11_peak_knee_flex_swing,
    12: rule_item12_peak_hip_ext,
    13: rule_item13_peak_hip_flex,
    16: lambda d, c: rule_item16_trunk_lean(d, c),
}

# Coronal rules
COR_RULES_SIDE_AWARE = {
    4:  rule_item4_hindfoot,              # fn(d_cor_side, side)
    5:  rule_item5_foot_rotation,         # fn(full, side)
    14: rule_item14_pelvic_obliquity,     # fn(full, side)
    15: rule_item15_pelvic_rotation,      # fn(full, side)
    17: rule_item17_lateral_trunk,        # fn(full, side)
}


def _process_sagittal_video(kps, scores, side):
    d = _get_sag_keypoints(kps, scores, side)
    if d is None:
        return {}
    hs, to = detect_gait_events_zeni(
        d['heel'][:, 0], d['toe'][:, 0], d['hip'][:, 0])
    cycles = extract_gait_cycles(hs, to, d['n'])

    out = {}
    for item_i, fn in SAG_RULES.items():
        try:
            res = fn(d, cycles)
        except Exception as e:
            res = None
        if res is None:
            continue
        binary, raw, ok = res
        if ok:
            out[item_i] = (binary, raw)
    return out


def _process_coronal_video(kps, scores, side):
    full = _extract_coronal_full(kps, scores)
    if full is None:
        return {}
    out = {}
    for item_i, fn in COR_RULES_SIDE_AWARE.items():
        try:
            if item_i == 4:
                d_cor = {
                    'knee':  full[f'{side}_knee'],
                    'ankle': full[f'{side}_ankle'],
                }
                res = fn(d_cor, side)
            else:
                res = fn(full, side)
        except Exception:
            res = None
        if res is None:
            continue
        binary, raw, ok = res
        if ok:
            out[item_i] = (binary, raw)
    return out


# --- Run extraction ---
all_pids = sorted(set(df_train['patient_id'].unique()) |
                  set(df_test['patient_id'].unique()))

records = []
raw_records = []
n_videos_used = 0
n_videos_skipped = 0

for pid in tqdm(all_pids, desc='EVGS rule sources'):
    if pid not in kp_data:
        n_videos_skipped += 1
        continue

    videos = kp_data[pid]
    side_item_binary = {
        'left':  {i: [] for i in range(1, 18)},
        'right': {i: [] for i in range(1, 18)},
    }
    side_item_raw = {
        'left':  {i: [] for i in range(1, 18)},
        'right': {i: [] for i in range(1, 18)},
    }

    for vname, vdata in videos.items():
        view = _get_video_view(vdata, vname)
        kps, sc = _get_video_arrays(vdata)
        if kps is None or len(kps) < MIN_FRAMES:
            n_videos_skipped += 1
            continue

        if view in ('left', 'right'):
            side = view
            res = _process_sagittal_video(kps, sc, side)
            for i, (b, r) in res.items():
                side_item_binary[side][i].append(b)
                side_item_raw[side][i].append(r)
            n_videos_used += 1
        elif view in ('forward', 'backward'):
            for side in ('left', 'right'):
                res = _process_coronal_video(kps, sc, side)
                for i, (b, r) in res.items():
                    side_item_binary[side][i].append(b)
                    side_item_raw[side][i].append(r)
            n_videos_used += 1
        else:
            n_videos_skipped += 1

    for side in ('left', 'right'):
        rec  = {'patient_id': pid, 'side': side}
        rrec = {'patient_id': pid, 'side': side}
        for i in range(1, 18):
            bs = side_item_binary[side][i]
            rs = [x for x in side_item_raw[side][i] if np.isfinite(x)]
            if bs:
                rec[f'rule_evgs_{i}'] = int(round(np.mean(bs)))
                rec[f'cov_evgs_{i}']  = len(bs)
                rrec[f'raw_evgs_{i}'] = float(np.median(rs)) if rs else np.nan
            else:
                rec[f'rule_evgs_{i}'] = -1
                rec[f'cov_evgs_{i}']  = 0
                rrec[f'raw_evgs_{i}'] = np.nan
        records.append(rec)
        raw_records.append(rrec)

df_rules = pd.DataFrame(records)
df_raw   = pd.DataFrame(raw_records)

print(f'\nRule extraction complete.')
print(f'  Videos used: {n_videos_used}')
print(f'  Videos skipped: {n_videos_skipped}')
print(f'  Output shape: {df_rules.shape}')

print(f'\nPer-item coverage:')
for i in range(1, 18):
    n = (df_rules[f'rule_evgs_{i}'] >= 0).sum()
    pct = 100 * n / len(df_rules)
    print(f'  Item {i:2d}: {n:3d}/{len(df_rules)} ({pct:5.1f}%)')

cell_end('Cell 5: Apply Rules to All Patients')


In [ ]:
# === Cell 6: Auto-Tune Binary Thresholds from Train Data ===
cell_start('Cell 6: Auto-Tune Thresholds')

# -------------------------------------------------------------
# For each (item, side) we have a raw continuous value in df_raw
# and a ground-truth binary label in df_train. We grid-search a
# 1D threshold AND a comparison mode (single interval / two-tail)
# to maximize per-item binary accuracy on the train set.
#
# Three tuning modes are tested per item:
#   (a) raw < thr_low OR raw > thr_high  -> abnormal=1  (two-tail)
#   (b) raw < thr                         -> abnormal=1  (one-sided low)
#   (c) raw > thr                         -> abnormal=1  (one-sided high)
# The best of the three is kept per item (and per side for items
# where the paper uses asymmetric thresholds: 4, 14).
# -------------------------------------------------------------

# Merge raw values with labels on (pid, side)
df_tune = df_train[['patient_id', 'side'] + LABEL_COLS].merge(
    df_raw, on=['patient_id', 'side'], how='left')
print(f'Tuning table: {df_tune.shape}')

PER_SIDE_ITEMS = {4, 14}  # items with side-asymmetric thresholds

def _eval_two_tail(raw, y, lo, hi):
    p = ((raw < lo) | (raw > hi)).astype(int)
    return (p == y).mean()

def _eval_one_low(raw, y, thr):
    p = (raw < thr).astype(int)
    return (p == y).mean()

def _eval_one_high(raw, y, thr):
    p = (raw > thr).astype(int)
    return (p == y).mean()


def _tune_item_side(raw_vals, y_vals):
    """Return (best_mode, params, best_acc, baseline)."""
    mask = np.isfinite(raw_vals)
    raw = raw_vals[mask]
    y   = y_vals[mask].astype(int)
    if len(raw) < 8:
        return None
    baseline = max((y == 0).mean(), (y == 1).mean())

    # Candidate thresholds = unique values + midpoints
    u = np.unique(raw)
    if len(u) > 80:
        u = np.linspace(u[0], u[-1], 80)
    # Use int(bool) directly to avoid numpy.bool_ rounding edge case
    maj_init = int(y.mean() >= 0.5)
    best = {'acc': -1.0, 'mode': 'const',
            'params': {'const': maj_init}}

    # (a) two-tail grid
    for lo in u:
        for hi in u:
            if hi <= lo:
                continue
            acc = _eval_two_tail(raw, y, lo, hi)
            if acc > best['acc']:
                best = {'acc': acc, 'mode': 'two_tail',
                        'params': {'lo': float(lo), 'hi': float(hi)}}

    # (b) one-sided low
    for thr in u:
        acc = _eval_one_low(raw, y, thr)
        if acc > best['acc']:
            best = {'acc': acc, 'mode': 'one_low',
                    'params': {'thr': float(thr)}}

    # (c) one-sided high
    for thr in u:
        acc = _eval_one_high(raw, y, thr)
        if acc > best['acc']:
            best = {'acc': acc, 'mode': 'one_high',
                    'params': {'thr': float(thr)}}

    # (d) constant (majority class)
    # Use int(bool) directly to avoid numpy.bool_ rounding edge case
    maj = int(y.mean() >= 0.5)
    acc = float((np.full_like(y, maj) == y).mean())
    if acc > best['acc']:
        best = {'acc': acc, 'mode': 'const',
                'params': {'const': maj}}

    best['baseline'] = baseline
    return best


tuned = {}  # {(item, side_or_"all"): best_dict}

for i in range(1, 18):
    raw_col = f'raw_evgs_{i}'
    lbl_col = f'evgs_{i}'
    if raw_col not in df_tune.columns:
        continue

    if i in PER_SIDE_ITEMS:
        for side in ('left', 'right'):
            sub = df_tune[df_tune['side'] == side]
            raw = sub[raw_col].values.astype(float)
            y = sub[lbl_col].values
            best = _tune_item_side(raw, y)
            if best is not None:
                tuned[(i, side)] = best
    else:
        raw = df_tune[raw_col].values.astype(float)
        y = df_tune[lbl_col].values
        best = _tune_item_side(raw, y)
        if best is not None:
            tuned[(i, 'all')] = best

print('\nTuned thresholds per item:')
print(f'{"Item":>4} {"Side":>5} {"Mode":>10} {"Params":>32} {"Acc":>7} {"BL":>7}')
print('-' * 70)
for (i, side), best in sorted(tuned.items()):
    p = best['params']
    pstr = ', '.join(f'{k}={v:.2f}' if isinstance(v, float) else f'{k}={v}'
                      for k, v in p.items())
    print(f'{i:>4d} {side:>5s} {best["mode"]:>10s} '
          f'{pstr:>32s} {best["acc"]:>7.4f} {best["baseline"]:>7.4f}')


def apply_tuned(i, side, raw):
    if not np.isfinite(raw):
        return -1
    key = (i, side) if i in PER_SIDE_ITEMS else (i, 'all')
    best = tuned.get(key)
    if best is None:
        return -1
    m = best['mode']
    p = best['params']
    if m == 'const':
        return int(p['const'])
    if m == 'two_tail':
        return int((raw < p['lo']) or (raw > p['hi']))
    if m == 'one_low':
        return int(raw < p['thr'])
    if m == 'one_high':
        return int(raw > p['thr'])
    return -1


# Build a new tuned-binary dataframe for all patients (train + test)
rec_tuned = []
for _, r in df_raw.iterrows():
    out = {'patient_id': int(r['patient_id']), 'side': r['side']}
    for i in range(1, 18):
        raw = r.get(f'raw_evgs_{i}', np.nan)
        out[f'tuned_evgs_{i}'] = apply_tuned(i, r['side'], raw)
    rec_tuned.append(out)
df_tuned = pd.DataFrame(rec_tuned)
print(f'\nTuned predictions table: {df_tuned.shape}')

cell_end('Cell 6: Auto-Tune Thresholds')


In [ ]:
# === Cell 7: Validate on Train (raw-rule + tuned-rule vs ML) ===
cell_start('Cell 7: Validate on Train')

df_train_eval = (df_train[['patient_id', 'side'] + LABEL_COLS]
                 .merge(df_rules, on=['patient_id', 'side'], how='left')
                 .merge(df_tuned, on=['patient_id', 'side'], how='left'))
print(f'Train eval: {df_train_eval.shape}')

# --- ML baseline via LOPO XGBoost on handcrafted features ---
try:
    import xgboost as xgb
    from sklearn.model_selection import LeaveOneGroupOut
    HAS_XGB = True
except Exception:
    HAS_XGB = False
print(f'xgboost available: {HAS_XGB}')

FEAT_COLS = sorted([c for c in df_train.columns
                    if c.startswith(('sag_', 'cor_', 'all_', 'n_'))])
X_tr = np.nan_to_num(df_train[FEAT_COLS].values.astype(np.float32), nan=0.0)
groups = df_train['patient_id'].values

ml_acc = {}
if HAS_XGB:
    print(f'Computing LOPO XGBoost OOF for ML baseline ({len(FEAT_COLS)} feats)...')
    logo = LeaveOneGroupOut()
    for item_i in tqdm(range(1, 18), desc='ML baseline'):
        y = df_train[f'evgs_{item_i}'].values.astype(int)
        oof = np.zeros_like(y)
        for tr_idx, te_idx in logo.split(X_tr, y, groups):
            try:
                clf = xgb.XGBClassifier(
                    n_estimators=200, max_depth=4, learning_rate=0.05,
                    subsample=0.9, colsample_bytree=0.9,
                    eval_metric='logloss', verbosity=0, n_jobs=2)
            except TypeError:
                clf = xgb.XGBClassifier(
                    n_estimators=200, max_depth=4, learning_rate=0.05,
                    subsample=0.9, colsample_bytree=0.9,
                    verbosity=0, n_jobs=2)
            clf.fit(X_tr[tr_idx], y[tr_idx])
            oof[te_idx] = clf.predict(X_tr[te_idx])
        ml_acc[item_i] = (oof == y).mean()
else:
    for item_i in range(1, 18):
        y = df_train[f'evgs_{item_i}'].values.astype(int)
        maj = int(round(np.mean(y) >= 0.5))
        ml_acc[item_i] = (y == maj).mean()

# --- Per-item comparison ---
rule_acc_raw = {}     # paper-threshold rule (Cell 5)
rule_acc_tuned = {}   # auto-tuned rule (Cell 6)
rule_cov = {}

for i in range(1, 18):
    col_true = f'evgs_{i}'
    col_raw  = f'rule_evgs_{i}'
    col_tun  = f'tuned_evgs_{i}'

    mask = df_train_eval[col_raw] != -1
    rule_cov[i] = mask.mean() * 100

    if mask.sum() == 0:
        rule_acc_raw[i] = np.nan
    else:
        y = df_train_eval.loc[mask, col_true].astype(int).values
        p = df_train_eval.loc[mask, col_raw].astype(int).values
        rule_acc_raw[i] = (y == p).mean()

    mask_t = df_train_eval[col_tun] != -1
    if mask_t.sum() == 0:
        rule_acc_tuned[i] = np.nan
    else:
        y = df_train_eval.loc[mask_t, col_true].astype(int).values
        p = df_train_eval.loc[mask_t, col_tun].astype(int).values
        rule_acc_tuned[i] = (y == p).mean()

print('\nPer-item train accuracy (raw rule / tuned rule / ML):')
print(f'{"Item":>4} {"Cov%":>6} {"Raw":>8} {"Tuned":>8} {"ML":>8} '
      f'{"d_tune":>8} {"d_raw":>8}')
print('-' * 60)
results_rows = []
for i in range(1, 18):
    r = rule_acc_raw[i]
    t = rule_acc_tuned[i]
    m = ml_acc[i]
    dt = t - m if (np.isfinite(t) and np.isfinite(m)) else np.nan
    dr = r - m if (np.isfinite(r) and np.isfinite(m)) else np.nan
    print(f'{i:>4d} {rule_cov[i]:>6.1f} {r:>8.4f} {t:>8.4f} {m:>8.4f} '
          f'{dt:>+8.4f} {dr:>+8.4f}')
    results_rows.append({
        'item': i,
        'coverage_pct': rule_cov[i],
        'raw_rule_acc': r,
        'tuned_rule_acc': t,
        'ml_acc': m,
        'delta_tuned_minus_ml': dt,
        'delta_raw_minus_ml': dr,
    })

df_compare = pd.DataFrame(results_rows)

tuned_wins = df_compare[df_compare['delta_tuned_minus_ml'] > 0]['item'].tolist()
tuned_high = df_compare[df_compare['tuned_rule_acc'] >= 0.85]['item'].tolist()
tuned_top5 = df_compare.dropna(subset=['delta_tuned_minus_ml']).sort_values(
                 'delta_tuned_minus_ml', ascending=False).head(5)['item'].tolist()

print(f'\nItems where tuned_rule > ML : {tuned_wins}')
print(f'Items with tuned_rule_acc >= 0.85: {tuned_high}')
print(f'Top-5 tuned_rule-vs-ML delta items: {tuned_top5}')

cell_end('Cell 7: Validate on Train')


In [ ]:
# === Cell 8: Generate Hybrid CSVs ===
cell_start('Cell 8: Generate Hybrid CSVs')

if df_base is None:
    print('\u26a0\ufe0f  No base CSV found -- skipping hybrid CSV generation.')
    generated = []
    cell_end('Cell 8: Generate Hybrid CSVs')
else:
    ITEM_COLS_L = [f'L{i}' for i in range(1, 18)]
    ITEM_COLS_R = [f'R{i}' for i in range(1, 18)]
    ITEM_COLS = ITEM_COLS_L + ITEM_COLS_R

    # Use TUNED predictions for the test patients
    test_preds = {}
    for _, row in df_tuned.iterrows():
        pid = int(row['patient_id'])
        if pid not in T1_TEST_IDS:
            continue
        side = row['side']
        test_preds[(pid, side)] = {
            i: int(row[f'tuned_evgs_{i}']) for i in range(1, 18)
        }
    print(f'Test patient-sides with tuned predictions: '
          f'{len(test_preds)} (expected {len(T1_TEST_IDS)*2})')

    def _preserve_track2_rows(df_new):
        # Track-2 rows are preserved from the current base CSV (overwritten by notebook 03).
        return df_new

    def _recompute_total(df_new):
        t1 = df_new['ID'].str.startswith('track1-')
        df_new.loc[t1, 'Total'] = df_new.loc[t1, ITEM_COLS].sum(axis=1).astype(int)
        return df_new

    def make_hybrid_csv(items_to_replace, suffix, description):
        df_new = df_base.copy()
        n_reviews = 0
        n_kept    = 0
        for pid in T1_TEST_IDS:
            rid = f'track1-{pid}'
            if (df_new['ID'] == rid).sum() == 0:
                continue
            for side in ('left', 'right'):
                side_letter = 'L' if side == 'left' else 'R'
                preds = test_preds.get((pid, side), {})
                for item_i in items_to_replace:
                    col = f'{side_letter}{item_i}'
                    val = preds.get(item_i, -1)
                    if val == -1:
                        n_kept += 1
                        continue
                    old_val = int(df_new.loc[df_new['ID'] == rid, col].values[0])
                    if old_val != val:
                        df_new.loc[df_new['ID'] == rid, col] = int(val)
                        n_reviews += 1
        df_new = _preserve_track2_rows(df_new)
        df_new = _recompute_total(df_new)
        # Route to canonical (read by final_assembly) vs diagnostic (offline review)
        is_canonical = (
            len(items_to_replace) == 1
            and items_to_replace[0] in CANONICAL_EVGS_ITEMS
            and suffix == f'individual_item_{items_to_replace[0]:02d}'
        )
        out_dir = SUBMIT_DIR if is_canonical else DIAGNOSTIC_DIR
        out_path = f'{out_dir}/evgs_rules_{suffix}.csv'
        df_new.to_csv(out_path, index=False)
        return out_path, n_reviews, n_kept, description

    generated = []

    all_items = list(range(1, 18))
    p, ch, kp, desc = make_hybrid_csv(all_items, 'tuned_all',
        'All 17 EVGS items overridden by tuned rules')
    generated.append((p, desc, ch, kp))

    if tuned_top5:
        p, ch, kp, desc = make_hybrid_csv(tuned_top5, 'tuned_top5',
            f'Top-5 tuned>ML items: {tuned_top5}')
        generated.append((p, desc, ch, kp))

    if tuned_high:
        p, ch, kp, desc = make_hybrid_csv(tuned_high, 'tuned_high_accuracy',
            f'High-acc tuned items: {tuned_high}')
        generated.append((p, desc, ch, kp))

    if tuned_wins:
        p, ch, kp, desc = make_hybrid_csv(tuned_wins, 'tuned_any_win',
            f'All items where tuned > ML: {tuned_wins}')
        generated.append((p, desc, ch, kp))

    # Items 11, 13, 14 focus overlay (validation-selected items)
    p, ch, kp, desc = make_hybrid_csv([11, 13, 14], 'tuned_11_13_14',
        'Focused source overlay on the three validation-selected EVGS items')
    generated.append((p, desc, ch, kp))

    for i in range(1, 18):
        suffix = f'individual_item_{i:02d}'
        p, ch, kp, desc = make_hybrid_csv([i], suffix,
            f'Item {i} alone overridden by tuned rule')
        generated.append((p, desc, ch, kp))

    print(f'\nGenerated {len(generated)} CSVs:')
    for p, desc, ch, kp in generated:
        print(f'  {os.path.basename(p):42s}  reviews={ch:3d}  fell_back={kp:3d}')
        print(f'      -> {p}')
        print(f'      {desc}')

    cell_end('Cell 8: Generate Hybrid CSVs')


In [ ]:
# === Cell 9: Summary ===
cell_start('Summary')

total_time = sum(t for t in _cell_times.values() if t is not None)
time_report = '\n'.join(
    f'  {n}: {t:.1f}s ({t/60:.1f}min)'
    for n, t in _cell_times.items() if t is not None)

def _fmt_item(row):
    return (f'  Item {int(row["item"]):2d}: cov={row["coverage_pct"]:5.1f}%  '
            f'raw={row["raw_rule_acc"]:.4f}  tuned={row["tuned_rule_acc"]:.4f}  '
            f'ml={row["ml_acc"]:.4f}  '
            f'd_tune={row["delta_tuned_minus_ml"]:+.4f}')

per_item_lines = '\n'.join(_fmt_item(r) for _, r in df_compare.iterrows())

csv_lines = '\n'.join(f'  {os.path.basename(p)}' for p, *_ in generated) if generated else '  (none)'

print(f"""
{'='*70}
SUMMARY - {NOTEBOOK_NAME}
{'='*70}

Execution time:
{time_report}
  Total: {total_time:.1f}s ({total_time/60:.1f}min)

Environment:
  Platform : {'Colab' if IN_COLAB else 'Local/RunPod'}
  ROOT     : {ROOT}

Per-item train accuracy (raw / tuned / ML):
{per_item_lines}

Highlight items
  Tuned > ML            : {tuned_wins}
  Tuned acc >= 0.85     : {tuned_high}
  Top-5 delta items     : {tuned_top5}

Generated submission CSVs ({len(generated)}):
{csv_lines}

Notes
  - Track 2 cells are copied from the clinical baseline.
  - Items with no rule data fall back to reference baseline for that patient-side.
  - Auto-tuned thresholds are selected via 1-D grid search on TRAIN labels;
    Cell 7 reports per-item LOSO CV accuracy for validation diagnostics.
  - Generated CSV artifacts for offline review:
    
{'='*70}
""")

cell_end('Summary')
